# Semana 10: Apuntes de la clase

**Gestión activa y pasiva, evaluación de desempeño y riesgo de mercado** (CFA L2, Portfolio Management: *Analysis of Active Portfolio Management* y *Measuring and Managing Market Risk*; CFA L1: ratios de desempeño de *Portfolio Risk and Return: Part II*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/10_desempeno_riesgo/clase10_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Cierra la unidad 2: lo que aprendimos a medir (riesgo, Sharpe, beta, alfa) se convierte en criterio para juzgar a un gestor y para fijar límites de riesgo. La unidad 2 (semanas 6 a 10) se evalúa en el examen final; la sección 9 es el mapa de repaso de la unidad.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas y símbolos que usamos esta semana; los de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| *Benchmark* | Índice de referencia | El portafolio contra el que se mide un gestor (SPY para un fondo de acciones de Estados Unidos). |
| *Buy and hold* | Comprar y mantener | No rebalancear nunca: los pesos derivan con los precios. |
| TE | *Tracking error* (riesgo activo) | Desviación estándar del retorno del fondo menos el del índice. |
| *Active share* | Participación activa | Fracción de la cartera que difiere del índice; mide cuán activo es un fondo. |
| $S_p$ | Ratio de Sharpe | Exceso de retorno por unidad de riesgo **total**: $(\bar R_p - R_f)/\sigma_p$. |
| $T_p$ | Ratio de Treynor | Exceso de retorno por unidad de riesgo **sistemático**: $(\bar R_p - R_f)/\beta_p$. |
| $\alpha_p$ | Alfa de Jensen | Retorno realizado menos el exigido por el CAPM; intercepto de la regresión en exceso. |
| $M^2$ | Medida de Modigliani y Modigliani | El Sharpe en puntos de retorno: $R_f + S_p\,\sigma_m - \bar R_m$. |
| IR | *Information ratio* (ratio de información) | Retorno activo sobre *tracking error*: el Sharpe de la apuesta activa. |
| IC | *Information coefficient* | Correlación entre los pronósticos del gestor y los resultados. |
| BR | *Breadth* (amplitud) | Número de apuestas independientes por año. |
| TC | *Transfer coefficient* | Fracción de la habilidad que las restricciones del mandato dejan implementar. |
| VaR | *Value at Risk* (valor en riesgo) | Pérdida mínima que se espera superar con probabilidad $1 - c$ en un horizonte dado. |
| ES o CVaR | *Expected shortfall* (*conditional VaR*) | Pérdida esperada dado que se supera el VaR. |
| $z_c$ | Cuantil normal | $z_{0.95} = 1.645$, $z_{0.99} = 2.326$. |
| *Drawdown* | Caída desde el máximo | Pérdida acumulada desde el último pico del valor del portafolio. |
| *Backtest* | Prueba retrospectiva | Simular una regla con datos pasados; para el VaR, contar cuántas veces la pérdida lo superó. |

## 1. Gestión pasiva y activa, y por qué rebalancear

**Pasiva:** replicar un índice con el menor costo y *tracking error*; su fundamento es el CAPM (el mercado es el tangente). **Activa:** desviarse del índice para superarlo después de costos; su fundamento es la existencia de alfas. **La aritmética de Sharpe (1991):** la suma de todos los gestores activos es el mercado, así que antes de costos el activo promedio empata con el índice y después de costos pierde. Un gestor solo gana si otro pierde.

**Rebalanceo.** Sin intervención, los pesos derivan hacia los activos que más subieron y el riesgo cambia. Con el 50/50 de la minera y la eléctrica, un año de auge (minera $+36$, eléctrica $+4$) mueve los pesos y el riesgo:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from utils.finanzas import (portafolio_dos_activos, sharpe, treynor, jensen, m2, descomponer_riesgo,
                            tracking_error_modelo, ratio_informacion, ley_fundamental, riesgo_activo_optimo,
                            anos_para_significancia, var_parametrico, es_parametrico, escalar_var, backtest)

mu_min, sig_min, mu_ele, sig_ele, rho = 12.0, 24.0, 8.0, 12.0, 0.25       # todo en %
w_inicial = 0.50
w_deriva = w_inicial * 1.36 / (w_inicial * 1.36 + (1 - w_inicial) * 1.04)
antes = portafolio_dos_activos(w_inicial, mu_min, mu_ele, sig_min, sig_ele, rho)
despues = portafolio_dos_activos(w_deriva, mu_min, mu_ele, sig_min, sig_ele, rho)
print(f"Peso de la minera: {w_inicial:.1%} -> {w_deriva:.1%} tras un ano de auge")
print(f"E(R): {antes[0]:.2f}% -> {despues[0]:.2f}% | sigma: {antes[1]:.2f}% -> {despues[1]:.2f}%")
print(f"Rebalancear a 50/50 con 120 de valor: vender {120 * (w_deriva - 0.5):.1f} de minera y comprar lo mismo de electrica")

Salida esperada: el peso de la minera pasa de 50 a 56.7 por ciento; el riesgo sube de 14.70 a 15.73 (el retorno esperado solo de 10.00 a 10.27). Rebalancear vende lo que subió y compra lo que quedó atrás. Dos reglas: **por calendario** (cada mes, trimestre o año) y **por bandas** (solo cuando un peso se aleja más de $\pm b$ puntos del objetivo; es el estándar institucional porque controla el riesgo con menos operaciones). Los costos (comisiones, impuestos sobre ganancias realizadas, margen de compraventa) piden bandas más anchas para los activos caros de negociar.

## 2. Las medidas de desempeño ajustadas por riesgo

$$S_p = \frac{\bar R_p - R_f}{\sigma_p} \qquad T_p = \frac{\bar R_p - R_f}{\beta_p} \qquad \alpha_p = \bar R_p - [R_f + \beta_p(\bar R_m - R_f)] \qquad M^2_p = R_f + S_p\,\sigma_m - \bar R_m$$

Sharpe y $M^2$ usan el riesgo **total**: son las medidas correctas cuando el fondo es toda la riqueza riesgosa del inversionista. Treynor y el alfa de Jensen usan el riesgo **sistemático**: son las correctas cuando el fondo es una parte de un portafolio diversificado. El ejemplo de la clase: dos fondos que ganaron 12 por ciento.

In [ ]:
rf, rm, sigma_m = 4.5, 10.0, 16.0
fondos = pd.DataFrame({"retorno": [12.0, 12.0, 10.0], "sigma": [18.0, 30.0, 16.0], "beta": [1.1, 0.9, 1.0]},
                      index=["Andino (P)", "Pacifico (Q)", "Mercado (M)"])
fondos["Sharpe"] = [sharpe(r, rf, s) for r, s in zip(fondos["retorno"], fondos["sigma"])]
fondos["Treynor"] = [treynor(r, rf, b) for r, b in zip(fondos["retorno"], fondos["beta"])]
fondos["alfa de Jensen"] = [jensen(r, rf, b, rm) for r, b in zip(fondos["retorno"], fondos["beta"])]
fondos["M2"] = [m2(r, rf, s, rm, sigma_m) for r, s in zip(fondos["retorno"], fondos["sigma"])]
print(fondos.round(3).to_string())
print("\nRanking por Sharpe: ", list(fondos["Sharpe"].sort_values(ascending=False).index))
print("Ranking por Treynor:", list(fondos["Treynor"].sort_values(ascending=False).index))

Salida esperada: Sharpe 0.417, 0.250 y 0.344; Treynor 6.82, 8.33 y 5.50; alfas $+1.45$, $+2.55$ y 0; $M^2$ $+1.17$, $-1.50$ y 0. Por riesgo total gana Andino y Pacífico ni siquiera alcanza al mercado; por riesgo sistemático gana Pacífico. No hay contradicción: responden preguntas distintas.

## 3. Por qué los rankings difieren: el riesgo de cada fondo

La descomposición de la semana 9, $\sigma_p^2 = \beta_p^2\sigma_m^2 + \sigma_\varepsilon^2$, explica el desacuerdo: Pacífico consigue su alfa asumiendo un riesgo idiosincrático enorme, que se diversifica dentro de una cartera grande pero no para quien lo tiene como única posición.

In [ ]:
for nombre, fila in fondos.iloc[:2].iterrows():
    d = descomponer_riesgo(fila["beta"], sigma_m, fila["sigma"])
    print(f"{nombre:13s} varianza {fila['sigma'] ** 2:6.0f} = sistematica {d['sigma_sistematica'] ** 2:6.1f} + idiosincratica {d['sigma_idiosincratica'] ** 2:6.1f} "
          f"| sigma_eps = {d['sigma_idiosincratica']:4.1f} | R2 = {d['r2']:.2f}")

Salida esperada: Andino $324 = 309.8 + 14.2$ ($\sigma_\varepsilon = 3.8$, $R^2 = 0.96$); Pacífico $900 = 207.4 + 692.6$ ($\sigma_\varepsilon = 26.3$, $R^2 = 0.23$). Un $R^2$ bajo tiene además otra consecuencia: el beta está mal estimado, y con él el Treynor y el alfa.

## 4. Retorno activo, *tracking error* y ratio de información

Frente a un índice $B$: retorno activo $R_A = R_p - R_B$; riesgo activo (TE) $\sigma_A$; ratio de información $IR = \bar R_A / \sigma_A$. Con el modelo de mercado y el mercado como índice, el riesgo activo se descompone en riesgo de factor y riesgo específico:

$$\sigma_A^2 = (\beta_p - 1)^2\,\sigma_m^2 + \sigma_\varepsilon^2$$

In [ ]:
for nombre, fila in fondos.iloc[:2].iterrows():
    d = descomponer_riesgo(fila["beta"], sigma_m, fila["sigma"])
    te = tracking_error_modelo(fila["beta"], sigma_m, d["sigma_idiosincratica"])
    print(f"{nombre:13s} retorno activo = {fila['retorno'] - rm:+.1f} | TE = {te:5.2f} "
          f"(factor {abs(fila['beta'] - 1) * sigma_m:.2f}, especifico {d['sigma_idiosincratica']:.2f}) | IR = {ratio_informacion(fila['retorno'], rm, te):.3f}")

Salida esperada: Andino TE 4.10 e IR 0.49; Pacífico TE 26.37 e IR 0.08. Los dos ganan 2 puntos al índice, pero Andino lo hace con una apuesta pequeña y Pacífico con una enorme. Un IR de 0.5 sostenido es excelente; la mediana de los fondos activos, después de costos, es negativa.

## 5. La ley fundamental de la gestión activa

$$IR \approx IC \times \sqrt{BR} \times TC \qquad\qquad \sigma_A^* = \frac{IR}{SR_B}\,\sigma_B \qquad\qquad SR_p^2 = SR_B^2 + IR^2$$

Poca habilidad ($IC$) aplicada muchas veces ($BR$) equivale a mucha habilidad aplicada pocas veces. Y el nivel óptimo de riesgo activo crece con el IR del gestor relativo al Sharpe del índice.

In [ ]:
print(f"IC 0.05 con 100 apuestas: IR = {ley_fundamental(0.05, 100):.2f} | IC 0.10 con 25 apuestas: IR = {ley_fundamental(0.10, 25):.2f}")
print(f"IC 0.04 con 200 apuestas: IR = {ley_fundamental(0.04, 200):.2f}  (item 4 de la clase)")
sr_b = (rm - rf) / sigma_m
print(f"\nCon IR = 0.30 frente al mercado (Sharpe {sr_b:.3f}, sigma {sigma_m}): riesgo activo optimo = {riesgo_activo_optimo(0.30, sr_b, sigma_m):.1f}%"
      f" | Sharpe total = {np.sqrt(sr_b ** 2 + 0.30 ** 2):.3f}")

Salida esperada: IR de 0.50 en los dos casos de la lámina (y 0.57 para el ítem 4); riesgo activo óptimo de 14.0 por ciento y Sharpe total de 0.456.

## 6. ¿Habilidad o suerte? El estadístico $t$ del alfa

Con $T$ años de datos, el error estándar del alfa es aproximadamente $\sigma_\varepsilon/\sqrt{T}$. Para $t \geq 2$ hacen falta $T \geq (2\sigma_\varepsilon/\hat\alpha)^2$ años.

In [ ]:
for nombre, alfa, sig_eps in [("Andino", 1.45, 3.8), ("Pacifico", 2.55, 26.3)]:
    print(f"{nombre:9s} alfa {alfa:+.2f} con sigma_eps {sig_eps:4.1f}: hacen falta {anos_para_significancia(alfa, sig_eps):5.0f} anos para t = 2")

Salida esperada: 27 años para Andino y unos 425 para Pacífico (426 con los valores redondeados de la lámina). Con cinco o diez años de historial, ninguno de los dos alfas es distinguible de cero; el mayor de los dos es el menos creíble. Sesgos que agravan el problema: supervivencia, selección, persistencia débil y alfa que es beta disfrazada (ARKK, semana 9). Un alfa sin su $t$ es una anécdota.

## 7. Valor en riesgo

$VaR_c$ es la pérdida mínima que se supera con probabilidad $1 - c$ en el horizonte. Paramétrico (normal): $VaR = z_c\,\sigma - \mu$. Histórico: el percentil $1 - c$ de los retornos observados. Monte Carlo: el percentil de miles de escenarios simulados. Escalado: con $\mu \approx 0$, $VaR_h = VaR_1\sqrt{h}$; con $\mu$ relevante, $\mu$ escala por $h$ y $\sigma$ por $\sqrt{h}$. El ES (o CVaR) es la pérdida esperada dado que se supera el VaR; bajo normalidad, $ES_c = \sigma\,\phi(z_c)/(1 - c) - \mu$.

Ejemplo de la clase: el portafolio tangente de la semana 7 ($E = 9.54$, $\sigma = 12.25$) con 1,000,000.

In [ ]:
mu_t, sig_t, monto = 9.54, 12.25, 1_000_000          # anuales, en %
for nivel in (0.95, 0.99):
    print(f"VaR {nivel:.0%} a un ano = {var_parametrico(mu_t, sig_t, nivel):5.2f}% del valor = {var_parametrico(mu_t / 100, sig_t / 100, nivel, monto):,.0f}")
print(f"ES 95% a un ano  = {es_parametrico(mu_t, sig_t, 0.95):.2f}%")
mu_mes, sig_mes = mu_t / 12, sig_t / np.sqrt(12)
print(f"\nA un mes: mu = {mu_mes:.3f}%, sigma = {sig_mes:.3f}% -> VaR 95% = {var_parametrico(mu_mes, sig_mes, 0.95):.2f}% = {var_parametrico(mu_mes / 100, sig_mes / 100, 0.95, monto):,.0f}")
print(f"Raiz del tiempo desde el VaR mensual (ignora mu): {escalar_var(var_parametrico(mu_mes, sig_mes, 0.95), 12):.2f}% frente al anual exacto de {var_parametrico(mu_t, sig_t, 0.95):.2f}%")
print(f"\nLa minera sola (12, 24): VaR 95% anual = {var_parametrico(12.0, 24.0, 0.95):.2f}%")

Salida esperada: VaR anual de 10.61 por ciento (106,100) al 95 y de 18.96 al 99; ES de 15.73; VaR mensual de 5.02 (50,200). La raíz del tiempo aplicada al VaR mensual da 17.4 contra 10.6 del cálculo exacto: con un $\mu$ anual de 9.5 puntos la regla sobreestima mucho, por eso solo vale para horizontes cortos. Y la minera sola tiene VaR de 27.5: diversificar lo bajó a 10.6.

**Limitaciones:** el VaR no es la pérdida máxima (para la cola, el ES); el paramétrico subestima colas gordas y correlaciones que suben en crisis; el histórico solo conoce las crisis de su ventana; ventana, confianza y horizonte deben coincidir para comparar dos VaR. **Complementos:** sensibilidades (beta, duración, delta), escenarios y pruebas de estrés (semana 14) y límites de riesgo (por posición, por VaR, *stop-loss*, por riesgo activo).

## 8. Respuestas a los ítems de la clase

**1. C.** Treynor $= (11 - 3)/0.8 = 10.0$. La B es el Sharpe ($8/20 = 0.40$); la A es el alfa de Jensen ($11 - [3 + 0.8 \times 6] = 3.2$). Las tres son medidas correctas; la pregunta pedía Treynor.

**2. B.** Si el fondo es toda la riqueza, el riesgo relevante es el total: Sharpe (o $M^2$). Treynor y alfa suponen que el riesgo idiosincrático se diversifica en el resto de la cartera, y aquí no hay resto.

**3. A.** $VaR = 1.645(20) - 8 = 32.9 - 8 = 24.9$ por ciento. La B olvida restar el retorno esperado; la C usa $z_{0.99}$ (2.326) con el mismo olvido corregido a medias: $2.326(20) - 8 = 38.5$.

**4. C.** $IR = 0.04 \times \sqrt{200} = 0.04 \times 14.14 = 0.57$. La A es el IC sin amplificar; la B multiplica por BR en lugar de por su raíz.

## 9. Mapa de repaso de la unidad 2 (semanas 6 a 10)

| Semana | Idea central | Fórmula clave | Nuestro ejemplo |
|---|---|---|---|
| 6. Riesgo y rendimiento | El riesgo de un portafolio no es el promedio de los riesgos; la correlación manda | $\sigma_p^2 = w_a^2\sigma_a^2 + w_b^2\sigma_b^2 + 2w_aw_b\rho\sigma_a\sigma_b$ | 50/50: $E = 10.0$, $\sigma = 14.7$ (promedio 18); PMV 12.5% en la minera |
| 7. Markowitz | Frontera eficiente, PMV, tangente, CAL y separación en dos fondos | $S_p = (E - R_f)/\sigma$; $y^* = (E_T - R_f)/(A\sigma_T^2)$ | Tangente 35.9/64.1: 9.44 y 12.91, Sharpe 0.382; $A = 4$: $y^* = 74\%$ |
| 9. CAPM y factores | Solo se paga el riesgo sistemático; el alfa depende del modelo | $E(R_i) = R_f + \beta_i[E(R_m) - R_f]$; $\sigma_i^2 = \beta_i^2\sigma_m^2 + \sigma_\varepsilon^2$ | Minera: exigido 11.10, alfa $+0.90$; $576 = 368.6 + 207.4$; FF3 13.10, alfa $-1.10$ |
| 10. Desempeño y VaR | Qué riesgo usar para juzgar depende del papel del fondo; el VaR es un umbral, no un máximo | $T_p = (\bar R_p - R_f)/\beta_p$; $\alpha_p$; $IR = \bar R_A/\sigma_A$; $VaR = z\sigma - \mu$ | Andino y Pacífico: Sharpe 0.417 contra 0.250, Treynor 6.82 contra 8.33; VaR 95% del tangente 10.61 |

## 10. Lista de verificación

Sabes de esta semana si puedes: explicar la aritmética de la gestión activa y qué miden el *tracking error* y el *active share*; mostrar con números por qué comprar y mantener cambia el riesgo y qué hace cada regla de rebalanceo; calcular Sharpe, Treynor, alfa de Jensen y $M^2$ y decir cuál corresponde a cada tipo de cliente; explicar con la descomposición del riesgo por qué dos medidas ordenan distinto; calcular un ratio de información y descomponer el riesgo activo; enunciar la ley fundamental y el nivel óptimo de riesgo activo; estimar cuántos años hacen falta para que un alfa sea significativo; calcular un VaR paramétrico a distintos horizontes y niveles, explicar el histórico y el Monte Carlo, y decir qué agrega el ES y qué no dice el VaR.